## RouteRight AI – Member 4 Data Processing Continuation Instructions

### Purpose of This Document

This document explains how Member 4 should continue and complete the preprocessing work started by Member 3.

Member 3 has completed all preprocessing operations that can safely be performed **before the train/test split**.

However, several preprocessing operations were intentionally left incomplete because they depend on information learned from the data distribution.

These remaining operations must only be fitted using the **training data after the train/test split**.

This is necessary to prevent information from the test set from influencing preprocessing decisions and causing **data leakage**.

Therefore, Member 4 is responsible for completing the remaining data-dependent preprocessing pipeline.

---

### 1. Starting Dataset

Do **not** restart preprocessing from the original event-log dataset.

Start from the dataset prepared by Member 3:

`Dataprocessing_part3/routeright_presplit_dataset.csv`

This is the official pre-split handoff dataset.

It contains:

* **24,918 incident records**
* **16 columns**
* **14 candidate model-input features**
* **1 chronological split-reference variable**
* **1 prediction target**

---

### 2. What Member 3 Already Completed

Member 3 has already completed:

* conversion of `?` placeholders to proper missing values,
* missing-value investigation,
* feature availability analysis,
* data-leakage analysis,
* constant and near-constant feature analysis,
* high-missingness feature analysis,
* removal of unsuitable and leakage-prone variables,
* datetime parsing,
* temporal feature engineering,
* categorical string cleaning,
* categorical cardinality review,
* ordinal feature encoding,
* removal of redundant original representations,
* creation of the final pre-split dataset,
* preprocessing documentation, and
* structural validation of the handoff dataset.

These steps should **not be repeated** by Member 4.

---

### 3. What Is Still Incomplete

The following preprocessing steps were intentionally deferred by Member 3:

1. categorical missing-value handling,
2. rare-category handling,
3. nominal categorical encoding,
4. unseen-category handling,
5. model-specific numerical scaling,
6. final train/test transformation, and
7. final model-ready feature generation.

These steps were **not forgotten**.

They were postponed because they depend on information learned from the data distribution.

They must therefore be performed **after the train/test split using the training data only**.

This is the main continuation responsibility of Member 4.

---

### 4. Final Candidate Model Inputs

#### Nominal Categorical Features

The following features still require categorical preprocessing:

* `opened_by`
* `contact_type`
* `location`
* `category`
* `subcategory`
* `u_symptom`
* `assignment_group`

---

#### Temporal Engineered Features

The following features were already created by Member 3:

* `opened_hour`
* `opened_day_of_week`
* `opened_month`
* `opened_is_weekend`

These features contain no missing values in the Member 3 handoff dataset.

---

#### Ordinal Encoded Features

The following features were already encoded by Member 3:

* `impact_encoded`
* `urgency_encoded`
* `priority_encoded`

Do **not** encode these variables again.

Their meanings are:

##### Impact

* 1 = Low
* 2 = Medium
* 3 = High

##### Urgency

* 1 = Low
* 2 = Medium
* 3 = High

##### Priority

* 1 = Low
* 2 = Moderate
* 3 = High
* 4 = Critical

---

### 5. Chronological Split Reference

The variable:

`opened_at_dt`

must be used only for creating the chronological train/test split.

It must **not** be included directly as a machine-learning predictor.

Its purpose is:

```text
Older incidents → Training data
Newer incidents → Testing data
```

The original date range is approximately:

* Earliest incident: `2016-02-29 01:16:00`
* Latest incident: `2017-02-16 14:17:00`

---

### 6. Prediction Target

The prediction target is:

`reassignment_required`

where:

* `0` = No Reassignment Required
* `1` = Reassignment Required

The original overall distribution is:

* 13,549 incidents with class `0`
* 11,369 incidents with class `1`

Approximately:

* 54.37% class `0`
* 45.63% class `1`

The target is reasonably balanced.

Therefore, SMOTE, oversampling, and undersampling are **not required initially**.

---

### 7. Most Important Rule

The correct preprocessing order is:

```text
Member 3 Pre-Split Dataset
            ↓
Chronological Train/Test Split
            ↓
        TRAIN              TEST
          │                  │
          ▼                  │
Fit preprocessing            │
using TRAIN ONLY             │
          │                  │
          ├─ Missing handling
          ├─ Rare-category rules
          ├─ One-hot encoder
          └─ Scaling if required
          │                  │
          ▼                  ▼
Transform TRAIN       Transform TEST
                 using TRAIN rules
          │                  │
          └─────────┬────────┘
                    ▼
           Model-Ready Data
```

Never learn preprocessing rules from the full dataset before splitting.

---

### 8. Load the Handoff Dataset

```python
import pandas as pd
import numpy as np

DATA_PATH = "Dataprocessing_part3/routeright_presplit_dataset.csv"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["opened_at_dt"]
)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())
```

Expected shape:

```text
(24918, 16)
```

---

### 9. Verify the Handoff Dataset

Before continuing, verify the basic structure.

```python
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing target values:")
print(df["reassignment_required"].isna().sum())

print("\nMissing split-reference values:")
print(df["opened_at_dt"].isna().sum())
```

Expected:

```text
Rows: 24918
Columns: 16

Missing target values:
0

Missing split-reference values:
0
```

---

### 10. Define Feature Groups

```python
nominal_features = [
    "opened_by",
    "contact_type",
    "location",
    "category",
    "subcategory",
    "u_symptom",
    "assignment_group"
]

temporal_features = [
    "opened_hour",
    "opened_day_of_week",
    "opened_month",
    "opened_is_weekend"
]

ordinal_features = [
    "impact_encoded",
    "urgency_encoded",
    "priority_encoded"
]

model_input_features = (
    nominal_features
    + temporal_features
    + ordinal_features
)

split_reference = "opened_at_dt"

target = "reassignment_required"
```

Verify:

```python
print("Nominal features:", len(nominal_features))
print("Temporal features:", len(temporal_features))
print("Ordinal features:", len(ordinal_features))
print("Total candidate predictors:", len(model_input_features))
```

Expected:

```text
Nominal features: 7
Temporal features: 4
Ordinal features: 3
Total candidate predictors: 14
```

---

### 11. Sort the Dataset Chronologically

Before creating the train/test split, sort the incidents by opening time.

```python
df = (
    df
    .sort_values("opened_at_dt")
    .reset_index(drop=True)
)
```

Verify:

```python
print("Earliest record:")
print(df["opened_at_dt"].iloc[0])

print("\nLatest record:")
print(df["opened_at_dt"].iloc[-1])
```

This ensures that historical incidents appear before newer incidents.

---

### 12. Create the Chronological Train/Test Split

Use a chronological split rather than a random split.

Recommended starting ratio:

* **80% oldest incidents → Training**
* **20% newest incidents → Testing**

```python
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
```

Approximately:

```text
Training rows: 19934
Testing rows: 4984
```

A small difference is acceptable if the team later decides to keep incidents with identical boundary timestamps together.

---

### 13. Validate the Chronological Split

Check the date ranges.

```python
print("Training period:")
print(
    train_df["opened_at_dt"].min(),
    "→",
    train_df["opened_at_dt"].max()
)

print("\nTesting period:")
print(
    test_df["opened_at_dt"].min(),
    "→",
    test_df["opened_at_dt"].max()
)
```

Then verify chronological separation:

```python
chronological_split_valid = (
    train_df["opened_at_dt"].max()
    <=
    test_df["opened_at_dt"].min()
)

print(
    "Chronological order valid:",
    chronological_split_valid
)
```

Expected:

```text
Chronological order valid: True
```

---

### 14. Check Target Distribution After Splitting

Do not use stratification if it breaks chronological order.

Instead, inspect the natural target distributions.

```python
print("Training target counts:")
print(
    train_df[target]
    .value_counts()
    .sort_index()
)

print("\nTraining target percentages:")
print(
    train_df[target]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("\nTesting target counts:")
print(
    test_df[target]
    .value_counts()
    .sort_index()
)

print("\nTesting target percentages:")
print(
    test_df[target]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)
```

A difference between training and test target distributions is not automatically an error.

It may represent a genuine change in incident behaviour over time.

---

### 15. Create `X_train`, `X_test`, `y_train`, and `y_test`

```python
X_train = train_df[
    model_input_features
].copy()

y_train = train_df[
    target
].copy()

X_test = test_df[
    model_input_features
].copy()

y_test = test_df[
    target
].copy()
```

Verify:

```python
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train length:", len(y_train))
print("y_test length:", len(y_test))
```

Important:

`opened_at_dt` must **not** appear in `X_train` or `X_test`.

Check:

```python
print(
    "opened_at_dt in X_train:",
    "opened_at_dt" in X_train.columns
)

print(
    "Target in X_train:",
    target in X_train.columns
)
```

Expected:

```text
opened_at_dt in X_train: False
Target in X_train: False
```

---

### 16. Complete Member 3's Remaining Missing-Value Handling

This is one of the main preprocessing tasks intentionally left for Member 4.

The remaining missing values occur in nominal categorical features.

The original full-dataset missing counts were:

| Feature            | Missing Count |
| ------------------ | ------------: |
| `u_symptom`        |         5,735 |
| `assignment_group` |           906 |
| `opened_by`        |           714 |
| `subcategory`      |            69 |
| `location`         |            52 |
| `category`         |            39 |

These values were intentionally preserved before the train/test split.

First inspect the missing values separately in training and testing data:

```python
print("Training missing values:")
display(
    X_train[nominal_features]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

print("Testing missing values:")
display(
    X_test[nominal_features]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)
```

---

#### Recommended Missing-Value Strategy

Represent missing categorical values using:

`"Missing"`

For example:

```text
NaN → Missing
```

Use:

```python
from sklearn.impute import SimpleImputer
```

Recommended imputer:

```python
SimpleImputer(
    strategy="constant",
    fill_value="Missing"
)
```

The imputer must be fitted using **training data only**.

---

### 17. Why Use a `"Missing"` Category?

For example, `u_symptom` contains a substantial number of missing values.

Replacing all missing symptoms with the most common symptom would incorrectly assume that those incidents actually belong to that category.

Using:

```text
Missing
```

preserves the fact that the information was unavailable.

Missingness itself may also contain useful predictive information.

---

### 18. Complete Member 3's Remaining Rare-Category Handling

Several nominal categorical variables have high cardinality.

Original approximate cardinalities were:

* `u_symptom` → 500 categories
* `subcategory` → 246 categories
* `location` → 219 categories
* `opened_by` → 207 categories
* `assignment_group` → 69 categories
* `category` → 57 categories
* `contact_type` → 5 categories

Some categories occur only a small number of times.

Rare-category handling should therefore be included before or during one-hot encoding.

However, category frequencies must be learned from the **training data only**.

Do not use the complete dataset to determine which categories are rare.

---

### 19. Inspect Training-Only Cardinality

```python
training_cardinality = pd.DataFrame({
    "Unique_Training_Categories": [
        X_train[feature].nunique(dropna=True)
        for feature in nominal_features
    ]
}, index=nominal_features)

training_cardinality = (
    training_cardinality
    .sort_values(
        "Unique_Training_Categories",
        ascending=False
    )
)

display(training_cardinality)
```

This inspection is safe because it only uses training data.

---

### 20. Recommended Rare-Category Strategy

A simple approach is to use scikit-learn's `OneHotEncoder` with infrequent-category handling.

```python
from sklearn.preprocessing import OneHotEncoder
```

Recommended starting configuration:

```python
OneHotEncoder(
    handle_unknown="infrequent_if_exist",
    min_frequency=10
)
```

This allows categories occurring fewer than 10 times in the training data to be treated as infrequent.

The exact threshold may later be adjusted if required, but any change must be justified.

If the installed scikit-learn version does not support:

```python
handle_unknown="infrequent_if_exist"
```

use:

```python
handle_unknown="ignore"
```

instead.

---

### 21. Complete Member 3's Remaining Nominal Encoding

The following nominal variables still require machine-learning encoding:

* `opened_by`
* `contact_type`
* `location`
* `category`
* `subcategory`
* `u_symptom`
* `assignment_group`

Use:

**One-Hot Encoding**

Do not use arbitrary integer encoding such as:

```text
Phone = 1
Email = 2
IVR = 3
```

because this would introduce a false numerical ordering.

---

### 22. Build the Nominal Preprocessing Pipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

nominal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=10
            )
        )
    ]
)
```

This pipeline completes four important preprocessing tasks:

1. categorical missing-value handling,
2. rare-category handling,
3. one-hot encoding,
4. unseen-category handling.

These are major preprocessing steps intentionally deferred by Member 3.

---

### 23. Numerical Features

The numerical candidate features are:

#### Temporal Features

* `opened_hour`
* `opened_day_of_week`
* `opened_month`
* `opened_is_weekend`

#### Ordinal Features

* `impact_encoded`
* `urgency_encoded`
* `priority_encoded`

These variables contain no missing values in the Member 3 handoff dataset.

---

### 24. Complete the Numerical Scaling Decision

Member 3 intentionally did not scale the numerical features.

This is because scaling requirements depend on the machine-learning algorithm.

#### Models That Generally Benefit from Scaling

Examples include:

* Logistic Regression
* K-Nearest Neighbors
* Support Vector Machine

Possible scaler:

```python
from sklearn.preprocessing import StandardScaler
```

#### Models That Generally Do Not Require Scaling

Examples include:

* Decision Tree
* Random Forest
* Gradient Boosting

Therefore, scaling should be handled inside the relevant model pipeline rather than permanently modifying the shared dataset.

---

### 25. Build the Basic Shared Preprocessor

For the initial preprocessing pipeline, categorical features can be transformed while numerical features are passed through unchanged.

```python
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "nominal",
            nominal_pipeline,
            nominal_features
        ),
        (
            "numeric",
            "passthrough",
            temporal_features + ordinal_features
        )
    ]
)
```

This is suitable as a common starting preprocessor.

Model-specific scaling can be added later where required.

---

### 26. Fit the Preprocessor Using Training Data Only

Correct:

```python
X_train_processed = (
    preprocessor.fit_transform(X_train)
)

X_test_processed = (
    preprocessor.transform(X_test)
)
```

This is extremely important.

The training data uses:

```python
fit_transform()
```

The test data uses only:

```python
transform()
```

The test set must never be used to learn preprocessing rules.

---

### 27. Never Fit the Test Data

Incorrect:

```python
X_test_processed = (
    preprocessor.fit_transform(X_test)
)
```

Do **not** do this.

This allows preprocessing rules to be learned from the test data and causes data leakage.

Correct:

```python
X_test_processed = (
    preprocessor.transform(X_test)
)
```

---

### 28. Never Preprocess the Full Dataset Before Splitting

Incorrect:

```python
X_all_processed = (
    preprocessor.fit_transform(
        df[model_input_features]
    )
)
```

before the train/test split.

The split must occur first.

Then the preprocessor must be fitted using `X_train`.

---

### 29. Validate the Final Processed Datasets

After preprocessing:

```python
print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed testing shape:",
    X_test_processed.shape
)

print(
    "Training target rows:",
    len(y_train)
)

print(
    "Testing target rows:",
    len(y_test)
)
```

Verify row consistency:

```python
print(
    "Training rows aligned:",
    X_train_processed.shape[0] == len(y_train)
)

print(
    "Testing rows aligned:",
    X_test_processed.shape[0] == len(y_test)
)
```

Expected:

```text
Training rows aligned: True
Testing rows aligned: True
```

The number of transformed features will probably be much greater than 14 because of one-hot encoding.

This is expected.

---

### 30. Retrieve Final Feature Names

Where supported:

```python
final_feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Final number of transformed features:",
    len(final_feature_names)
)
```

Display some of them:

```python
print(final_feature_names[:30])
```

These names are useful for:

* feature interpretation,
* reporting,
* model debugging,
* feature importance analysis.

---

### 31. Save the Final Feature Names

```python
pd.DataFrame({
    "Feature": final_feature_names
}).to_csv(
    "processed_feature_names.csv",
    index=False
)

print(
    "Processed feature names saved successfully."
)
```

---

### 32. Validate Remaining Missing Values After Transformation

After preprocessing, there should be no untreated categorical missing values.

If the transformed matrices are sparse matrices, check the data values appropriately.

For example:

```python
import numpy as np

print(
    "NaN values in transformed training data:",
    np.isnan(
        X_train_processed.data
        if hasattr(X_train_processed, "data")
        else X_train_processed
    ).sum()
)

print(
    "NaN values in transformed testing data:",
    np.isnan(
        X_test_processed.data
        if hasattr(X_test_processed, "data")
        else X_test_processed
    ).sum()
)
```

Expected:

```text
NaN values in transformed training data: 0
NaN values in transformed testing data: 0
```

---

### 33. Additional Feature Validation – Priority Redundancy

Member 3 identified one additional feature-validation issue.

The features:

* `impact_encoded`
* `urgency_encoded`
* `priority_encoded`

appear strongly related.

Priority may be determined from the combination of impact and urgency.

This is **not target leakage**.

However, it may introduce redundant information, particularly for linear models.

Check the combinations using **training data only**:

```python
priority_check = (
    train_df[
        [
            "impact_encoded",
            "urgency_encoded",
            "priority_encoded"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "impact_encoded",
            "urgency_encoded"
        ]
    )
)

display(priority_check)
```

Document whether `priority_encoded` appears fully determined by the combination of `impact_encoded` and `urgency_encoded`.

Do not automatically remove it without analysis.

If required during model development, compare performance:

```text
Model A:
impact + urgency + priority

vs

Model B:
impact + urgency only
```

The final choice should be justified using model performance and interpretability.

---

### 34. Additional Feature Validation – Temporal Variables

The following variables:

* `opened_hour`
* `opened_day_of_week`
* `opened_month`

are numerical representations of time-related information.

For tree-based models, they can initially be used directly.

For linear models, alternative representations may later be investigated, such as:

* one-hot encoding,
* cyclical encoding.

For example, hour 23 and hour 0 are close in real time even though their numerical values are far apart.

This is a model-development refinement.

It does **not** mean the temporal feature engineering performed by Member 3 was incorrect.

---

### 35. Class Resampling Decision

Do not automatically apply:

* SMOTE,
* random oversampling,
* random undersampling.

The original target distribution is approximately:

```text
54% vs 46%
```

which is reasonably balanced.

Start with baseline models first.

If later model evaluation identifies class-specific performance problems, techniques such as:

* class weighting,
* oversampling,
* undersampling,
* SMOTE

may be investigated.

If any resampling technique is used, it must be applied **only to the training data**.

Never resample before creating the train/test split.

---

### 36. Duplicate Rows

Member 3 verified the original incident-level dataset using the incident identifier.

The dataset contained:

* 24,918 rows,
* 24,918 unique incidents,
* zero duplicate incident identifiers.

After removing the incident identifier, some different incidents may have identical final feature values.

Therefore, do **not** automatically remove rows simply because their final predictor values are identical.

They may represent separate valid incidents.

---

### 37. Final Leakage Checklist

Before completing the preprocessing stage, verify:

* [ ] Dataset started from `routeright_presplit_dataset.csv`
* [ ] Dataset was sorted using `opened_at_dt`
* [ ] Train/test split was created before learned preprocessing
* [ ] Older incidents are in the training set
* [ ] Newer incidents are in the testing set
* [ ] `opened_at_dt` is not used as a model predictor
* [ ] `reassignment_required` is not present in model inputs
* [ ] Missing-value handling was fitted using training data only
* [ ] Rare-category rules were learned using training data only
* [ ] One-hot encoder was fitted using training data only
* [ ] Test data uses only `transform()`
* [ ] Unknown test categories can be safely handled
* [ ] Scaling, if used, is fitted using training data only
* [ ] No SMOTE or other resampling was applied before splitting
* [ ] Training feature rows match `y_train`
* [ ] Testing feature rows match `y_test`
* [ ] No untreated missing values remain after transformation
* [ ] Final transformed feature names were recorded
* [ ] Priority redundancy was reviewed and documented

---

### 38. Things Member 4 Must Not Do

Do **not**:

1. restart preprocessing from the raw event log,
2. redo Member 3 feature removal,
3. encode `impact`, `urgency`, and `priority` again,
4. use `opened_at_dt` directly as a model predictor,
5. perform learned preprocessing before splitting,
6. calculate rare-category frequencies using the full dataset,
7. fit an imputer using test data,
8. fit the one-hot encoder using test data,
9. call `fit_transform()` on `X_test`,
10. fit scaling using test data,
11. apply SMOTE before the train/test split,
12. add removed future/lifecycle variables back into the model,
13. automatically remove valid incidents because their final feature values are identical,
14. include the target in any predictor transformation.

---

### 39. Recommended Member 4 Notebook Structure

The Member 4 notebook can follow this structure:

#### 1. Title and Objective

#### 2. Import Libraries

#### 3. Load Member 3 Handoff Dataset

#### 4. Verify Handoff Structure

#### 5. Define Feature Groups

#### 6. Sort Dataset Chronologically

#### 7. Create Chronological Train/Test Split

#### 8. Validate Chronological Separation

#### 9. Check Train/Test Target Distribution

#### 10. Create `X_train`, `X_test`, `y_train`, `y_test`

#### 11. Inspect Remaining Training/Test Missing Values

#### 12. Define Missing-Value Handling

#### 13. Inspect Training-Only Cardinality

#### 14. Define Rare-Category Handling

#### 15. Define Nominal Categorical Encoding

#### 16. Define Numerical Feature Handling

#### 17. Build Leakage-Safe `ColumnTransformer`

#### 18. Fit Preprocessor Using Training Data Only

#### 19. Transform Training and Test Data

#### 20. Validate Processed Feature Matrices

#### 21. Retrieve Final Feature Names

#### 22. Validate Remaining Missing Values

#### 23. Review Priority Redundancy

#### 24. Document Model-Specific Scaling Requirements

#### 25. Final Leakage Validation

#### 26. Save the Preprocessing Pipeline

#### 27. Final Summary

---

### 40. Save the Fitted Preprocessing Pipeline

The fitted preprocessing pipeline should ideally be saved so that the same transformations can be reused during model development and deployment.

```python
import joblib

joblib.dump(
    preprocessor,
    "routeright_preprocessor.joblib"
)

print(
    "Preprocessing pipeline saved successfully."
)
```

This allows later stages to apply exactly the same preprocessing rules.

---

### 41. Expected Member 4 Outputs

Member 4 should ideally provide:

#### Notebook

`04_TrainTest_PreprocessingPipeline.ipynb`

#### Fitted Preprocessing Pipeline

`routeright_preprocessor.joblib`

#### Final Transformed Feature Names

`processed_feature_names.csv`

These outputs should then be used by the model-development stage.

---

### 42. Responsibility Boundary

#### Member 3 Completed

Member 3 completed everything that could safely be performed before the train/test split:

```text
Missing placeholder conversion
        ↓
Feature availability review
        ↓
Data-leakage analysis
        ↓
Feature removal
        ↓
Datetime parsing
        ↓
Temporal feature engineering
        ↓
Categorical deterministic cleaning
        ↓
Ordinal encoding
        ↓
Pre-split feature finalization
        ↓
routeright_presplit_dataset.csv
```

---

#### Member 4 Completes

Member 4 completes the remaining data-dependent preprocessing:

```text
Chronological train/test split
        ↓
Training-only missing-value handling
        ↓
Training-only rare-category handling
        ↓
Training-only one-hot encoding
        ↓
Unseen-category handling
        ↓
Model-specific scaling where required
        ↓
Final leakage-safe
X_train and X_test
```

---

### 43. Complete Project Preprocessing Flow

```text
MEMBER 3
────────────────────────────────────
Incident-Level Dataset
        ↓
Convert ? → NaN
        ↓
Feature Availability Review
        ↓
Leakage Analysis
        ↓
Remove Unsuitable Features
        ↓
Datetime Feature Engineering
        ↓
Categorical Deterministic Cleaning
        ↓
Ordinal Encoding
        ↓
14 Candidate Predictors
        ↓
routeright_presplit_dataset.csv
────────────────────────────────────
                ↓
             HANDOFF
                ↓
MEMBER 4
────────────────────────────────────
Sort Chronologically
        ↓
Create Train/Test Split
        ↓
        ┌─────────────────────────┐
        │                         │
      TRAIN                     TEST
        │                         │
        ▼                         │
Fit Missing Handling              │
        ↓                         │
Fit Rare-Category Rules           │
        ↓                         │
Fit One-Hot Encoder               │
        ↓                         │
Fit Scaling if Required           │
        │                         │
        ▼                         ▼
Transform TRAIN            Transform TEST
        │             using TRAIN-fitted rules
        │                         │
        └────────────┬────────────┘
                     ▼
          Leakage-Safe ML Dataset
                     ↓
               Model Development
```

---

### 44. Final Main Rule

**Member 3 intentionally stopped before preprocessing operations that could learn information from the future test set.**

Therefore, Member 4 must:

> **Split the dataset first, fit all remaining data-dependent preprocessing using the training data only, and then apply those fitted preprocessing rules unchanged to the test data.**

This completes the preprocessing workflow started by Member 3 while preventing train/test data leakage.
